In [1]:
import numpy as np
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.metrics import mean_squared_error

from data import CaseVFL

In [2]:
N = 1000
feature_count = 100

rng = np.random.default_rng(42)
case = CaseVFL(
    rng = rng,
    n = N,
    events = feature_count,
    output_arity = 50,
    sensors = 2,
    sensor_arity = 50,
    noise_std = 0.01
)

X_train, Y_train = case.generate_data()
X_test, Y_test = case.generate_data()
X_train_agg = np.block([x.T for x in X_train])
X_test_agg = np.block([x.T for x in X_test])

# Baseline

In [3]:
model = Ridge(alpha=0.5).fit(X_train_agg, Y_train)
Y_train_pred = model.predict(X_train_agg)
print(f"Train Error: {mean_squared_error(Y_train_pred, Y_train)}")
Y_test_pred = model.predict(X_test_agg)
print(f"Test Error: {mean_squared_error(Y_test_pred, Y_test)}")

Train Error: 9.110656181217162e-05
Test Error: 0.00011959060203509258


# VFL

In [12]:
parties = 2
weights = [rng.uniform(size=(50)) for _ in range(parties)]

In [16]:
for iteration in range(100000):
    Y_hat = np.sum([(weights[i] @ X_train[i]) for i in range(parties)], axis=0)
    eta = 1 / 10000
    for i in range(parties):
        weights[i] -= (X_train[i] @ (Y_hat - Y_train)) * 2 / N * eta

In [19]:
Y_train_hat = np.sum([(weights[i] @ X_train[i]) for i in range(parties)], axis=0)
print("Train", mean_squared_error(Y_train_hat, Y_train))
Y_test_hat = np.sum([(weights[i] @ X_test[i]) for i in range(parties)], axis=0)
print("Test", mean_squared_error(Y_test_hat, Y_test))

Train 0.00041750433794632643
Test 0.0005319427146029642
